# **BilSTM**

BiLSTM — это двунаправленная LSTM: она смотрит на последовательность и слева-направо, и справа-налево одновременно.

По сути это “LSTM с двумя мозгами”:

первый читает ряд как обычно: от прошлого к будущему;

второй — наоборот: от будущего к прошлому;

их состояния склеиваются и подаются дальше

hₜ = [ h⃗ₜ ; h⃖ₜ ]


h⃗ₜ — LSTM вперёд по времени

h⃖ₜ — LSTM назад

[ ; ] — конкатенация

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [ ]:
import numpy as np
import pandas as pd
from preprocessing.preprocess import prep
from preprocessing.target import ttp_target
from metrics.Metrics import merged_metrics
name = "AFKS"
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

...

In [ ]:
# Берём только числовые колонки
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Что точно НЕ идёт в признаки
drop_feature_cols = ["Close_fwd", "ret_H", "GoodTrade"]

feature_cols = [c for c in numeric_cols if c not in drop_feature_cols]
print("Фичи для BiLSTM:", feature_cols)

# Сплит по времени на train / test
split_bar = int(len(df) * 0.8)
train_df = df.iloc[:split_bar].copy()
test_df  = df.iloc[split_bar:].copy()

# Масштабируем ТОЛЬКО по train
scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

print("Train bars:", len(train_df), "Test bars:", len(test_df))
print("Доля GoodTrade=1 в train:", train_df["GoodTrade"].mean())
print("Доля GoodTrade=1 в test :", test_df["GoodTrade"].mean())


Фичи для BiLSTM: ['Open', 'High', 'Low', 'Close', 'Alligator_Jaw', 'Alligator_Teeth', 'Alligator_Lips', 'Fractal_Up', 'Fractal_Down', 'AO', 'Color AO', 'Alligator_Bullish', 'Alligator_Bearish', 'AlligatorStart_Long', 'AlligatorStart_Short', 'AO_sign', 'AO_zero_up', 'AO_zero_down', 'AO_three_green', 'AO_three_red', 'AO_saucer_up', 'AO_saucer_down', 'EntrySignal', 'Fractal_Up_conf', 'Fractal_Down_conf', 'AddOn_Anchor_Level', 'AddOn_Anchor_IsUp', 'AddOn_Size_Pct', 'AddOn_Ready', 'AddOn_Triggered']
Train bars: 28979 Test bars: 7245
Доля GoodTrade=1 в train: 0.02425894613340695
Доля GoodTrade=1 в test : 0.023740510697032435


In [ ]:
SEQ_LEN = 50  # длина окна истории, можно изменить

def make_sequences(df_part, feature_cols, seq_len=50):
    data = df_part[feature_cols].values
    targets = df_part["GoodTrade"].values
    signals = df_part["EntrySignal"].values

    X_list, y_list = [], []

    for i in range(seq_len - 1, len(df_part)):
        # нас интересуют только бары, где есть сигнал
        if signals[i] == 0:
            continue

        X_seq = data[i - seq_len + 1 : i + 1, :]  # [seq_len, num_features]
        y_val = targets[i]

        X_list.append(X_seq)
        y_list.append(y_val)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    return X, y

X_train, y_train = make_sequences(train_df, feature_cols, seq_len=SEQ_LEN)
X_test, y_test   = make_sequences(test_df,  feature_cols, seq_len=SEQ_LEN)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("Доля GoodTrade=1 в train seq:", y_train.mean())
print("Доля GoodTrade=1 в test seq :", y_test.mean())


X_train: (28930, 50, 30) X_test: (7196, 50, 30)
Доля GoodTrade=1 в train seq: 0.024230902177670238
Доля GoodTrade=1 в test seq : 0.02376320177876598


In [ ]:
num_features = X_train.shape[2]

model = Sequential([
    Bidirectional(
        LSTM(64, return_sequences=False),
        input_shape=(SEQ_LEN, num_features)
    ),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")  # бинарная классификация
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 40s 78ms/step - accuracy: 0.9573 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 2/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 26s 58ms/step - accuracy: 0.9741 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 3/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 27s 59ms/step - accuracy: 0.9779 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 4/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 40s 58ms/step - accuracy: 0.9748 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 5/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 41s 58ms/step - accuracy: 0.9759 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 6/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 41s 58ms/step - accuracy: 0.9759 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 7/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 26s 58ms/step - accuracy: 0.9757 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 8/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 27s 60ms/step - accuracy: 0.9760 - loss: nan - val_accuracy: 0.9762 - val_loss: nan


In [ ]:
y_proba = model.predict(X_test).ravel()
y_pred = (y_proba >= 0.5).astype(int)

print("BiLSTM AUC:", roc_auc_score(y_test, y_proba))
print("\nОтчёт по классификации (BiLSTM):")
print(classification_report(y_test, y_pred))
print("Матрица ошибок (BiLSTM):")
print(confusion_matrix(y_test, y_pred))


225/225 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step


ValueError: Input contains NaN.